In [1]:
# Run raw data cell at bottom of notebook first.
import requests
import pandas as pd
import pyarrow

In [4]:
samples = [s.strip() for s in raw.splitlines() if s.strip()]
tumor = sum(1 for s in samples if s.split("-")[3].startswith("01"))
normal = sum(1 for s in samples if s.split("-")[3].startswith("11"))
other = len(samples) - tumor - normal
other_samples = [s for s in samples if not s.split("-")[3].startswith("01") and not s.split("-")[3].startswith("11")]
print(f"Other Samples: {other_samples}")
print(f"Tumor: {tumor}, Normal: {normal}, Other: {other}, Total: {len(samples)}")

Other Samples: ['TCGA-DV-A4W0-05A']
Tumor: 322, Normal: 160, Other: 1, Total: 483


In [5]:
URL = "https://gdc-hub.s3.us-east-1.amazonaws.com/download/TCGA-KIRC.methylation450.tsv.gz"
outpath = r"C:\Users\JoshK\OneDrive\Desktop\ML_Projects\dna_methylation_cancer_prediction\DATA\raw\TCGA-KIRC.methylation450.tsv.gz"
with requests.get(URL, stream=True) as response:
    response.raise_for_status()
    with open(outpath, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024*1024):
            file.write(chunk)


In [6]:
data = pd.read_csv(outpath, sep="\t", index_col=0)
data = data.astype("float32") # Save memory by converting to float32
data = data.drop(columns='TCGA-DV-A4W0-05A')
print(data.shape)
print(data.index[:5])
print(data.columns[:5])
data.head()

(486427, 482)
Index(['cg00000029', 'cg00000108', 'cg00000109', 'cg00000165', 'cg00000236'], dtype='str', name='Composite Element REF')
Index(['TCGA-B4-5834-01A', 'TCGA-DV-5575-01A', 'TCGA-B0-5108-01A',
       'TCGA-B0-5703-01A', 'TCGA-CJ-4905-11A'],
      dtype='str')


,TCGA-B4-5834-01A,TCGA-DV-5575-01A,TCGA-B0-5108-01A,TCGA-B0-5703-01A,TCGA-CJ-4905-11A,TCGA-B0-4714-01A,TCGA-B2-5635-01A,TCGA-B2-5635-01B,TCGA-B0-5698-01A,TCGA-BP-5195-11A,...,TCGA-BP-5010-11A,TCGA-B0-4824-01A,TCGA-B0-4823-11A,TCGA-A3-A6NI-01A,TCGA-CJ-5689-01A,TCGA-BP-4993-01A,TCGA-CZ-4865-01A,TCGA-BP-4782-11A,TCGA-B0-4945-11A,TCGA-B4-5843-01A
Composite Element REF,,,,,,,,,,,,,,,,,,,,,
cg00000029,0.614927,0.321158,0.450301,0.816181,0.387471,0.755179,0.547720,0.483266,0.632484,0.370276,...,0.411018,0.509449,0.398563,0.410223,0.584544,0.461124,0.594906,0.488778,0.528901,0.490657
cg00000108,0.972071,0.964997,0.972311,0.964033,0.960496,0.959082,0.971277,0.961411,0.969599,0.972698,...,0.967596,0.948776,0.967950,0.955545,0.975825,0.965083,0.969403,0.959676,0.970876,0.969888
cg00000109,0.612456,0.758764,0.678896,0.723863,0.860965,NaN,0.619517,0.621427,0.441777,0.927009,...,0.942418,0.797796,0.923463,0.582375,0.789440,0.599614,0.687554,0.891868,0.923883,0.519000
cg00000165,0.188422,0.130063,0.197717,0.152583,0.203964,0.131952,0.199536,0.250376,0.099765,0.130408,...,0.167742,0.154676,0.164576,0.409074,0.103965,0.163763,0.343633,0.185912,0.124858,0.104241
cg00000236,0.917789,0.931517,0.909915,0.874732,0.907699,0.892050,0.933855,0.932204,0.926848,0.888925,...,0.910573,0.899164,0.911225,0.910482,0.903064,0.926505,0.915849,0.936125,0.924377,0.948324


In [ ]:
labels = pd.Series(
    {s: "tumor" if s.split("-")[3].startswith("01")
     else "normal" if s.split("-")[3].startswith("11")
     else "other" for s in data.columns},
     name = "sample_type")
print(labels.value_counts())
(labels).head()
labels.to_csv(r"C:\Users\JoshK\OneDrive\Desktop\ML_Projects\dna_methylation_cancer_prediction\DATA\processed\sample_labels.csv")


sample_type
tumor     322
normal    160
Name: count, dtype: int64


TCGA-B4-5834-01A     tumor
TCGA-DV-5575-01A     tumor
TCGA-B0-5108-01A     tumor
TCGA-B0-5703-01A     tumor
TCGA-CJ-4905-11A    normal
Name: sample_type, dtype: str

False


In [8]:
data = data.T
# print(data.head())
print(data.shape)
print(data.isna().sum().sum(), "total missing values")
print(data.isna().mean().mean() * 100, "% missing overall")
data.to_parquet(r"C:\Users\JoshK\OneDrive\Desktop\ML_Projects\dna_methylation_cancer_prediction\DATA\processed\transposed_beta_values.parquet", engine="pyarrow")

(482, 486427)
35026891 total missing values
14.939528097792465 % missing overall


In [3]:
# Checking Ratio from Xena: https://xenabrowser.net/datapages/?host=https%3A%2F%2Fgdc.xenahubs.net&dataset=TCGA-KIRC.methylation450.tsv&allSamples=true&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443
raw = """TCGA-3Z-A93Z-01A
TCGA-6D-AA2E-01A
TCGA-A3-3357-01A
TCGA-A3-3357-11A
TCGA-A3-3358-01A
TCGA-A3-3367-01A
TCGA-A3-3367-11A
TCGA-A3-3370-01A
TCGA-A3-3370-11A
TCGA-A3-3373-01A
TCGA-A3-3373-11A
TCGA-A3-3376-01A
TCGA-A3-3376-11A
TCGA-A3-3385-01A
TCGA-A3-3385-11A
TCGA-A3-3387-01A
TCGA-A3-A6NI-01A
TCGA-A3-A6NJ-01A
TCGA-A3-A6NL-01A
TCGA-A3-A6NN-01A
TCGA-A3-A8CQ-01A
TCGA-A3-A8OU-01A
TCGA-A3-A8OV-01A
TCGA-A3-A8OW-01A
TCGA-A3-A8OX-01A
TCGA-AK-3425-01A
TCGA-AK-3428-01A
TCGA-AK-3431-01A
TCGA-AK-3433-01A
TCGA-AK-3434-01A
TCGA-AK-3440-01A
TCGA-AK-3445-01A
TCGA-AK-3450-01A
TCGA-AK-3453-01A
TCGA-AK-3454-01A
TCGA-AK-3458-01A
TCGA-AK-3460-01A
TCGA-AK-3461-01A
TCGA-B0-4688-01A
TCGA-B0-4688-11A
TCGA-B0-4690-01A
TCGA-B0-4690-11A
TCGA-B0-4691-01A
TCGA-B0-4691-11A
TCGA-B0-4693-01A
TCGA-B0-4693-11A
TCGA-B0-4694-01A
TCGA-B0-4694-11A
TCGA-B0-4696-01A
TCGA-B0-4696-11A
TCGA-B0-4697-01A
TCGA-B0-4697-11A
TCGA-B0-4698-01A
TCGA-B0-4698-11A
TCGA-B0-4699-01A
TCGA-B0-4699-11A
TCGA-B0-4700-01A
TCGA-B0-4701-01A
TCGA-B0-4701-11A
TCGA-B0-4703-01A
TCGA-B0-4703-11A
TCGA-B0-4706-01A
TCGA-B0-4706-11A
TCGA-B0-4707-01A
TCGA-B0-4707-11A
TCGA-B0-4710-01A
TCGA-B0-4710-11A
TCGA-B0-4712-01A
TCGA-B0-4712-11A
TCGA-B0-4713-01A
TCGA-B0-4713-11A
TCGA-B0-4714-01A
TCGA-B0-4714-11A
TCGA-B0-4718-01A
TCGA-B0-4718-11A
TCGA-B0-4810-01A
TCGA-B0-4810-11A
TCGA-B0-4811-01A
TCGA-B0-4811-11A
TCGA-B0-4813-01A
TCGA-B0-4813-11A
TCGA-B0-4814-01A
TCGA-B0-4814-11A
TCGA-B0-4815-01A
TCGA-B0-4815-11A
TCGA-B0-4816-01A
TCGA-B0-4816-11A
TCGA-B0-4817-01A
TCGA-B0-4817-11A
TCGA-B0-4818-01A
TCGA-B0-4818-11A
TCGA-B0-4819-01A
TCGA-B0-4819-11A
TCGA-B0-4821-01A
TCGA-B0-4821-11A
TCGA-B0-4822-01A
TCGA-B0-4822-11A
TCGA-B0-4823-01A
TCGA-B0-4823-11A
TCGA-B0-4824-01A
TCGA-B0-4824-11A
TCGA-B0-4827-01A
TCGA-B0-4827-11A
TCGA-B0-4828-01A
TCGA-B0-4828-11A
TCGA-B0-4841-01A
TCGA-B0-4841-11A
TCGA-B0-4842-01A
TCGA-B0-4842-11A
TCGA-B0-4843-01A
TCGA-B0-4843-11A
TCGA-B0-4844-01A
TCGA-B0-4844-11A
TCGA-B0-4845-01A
TCGA-B0-4845-11A
TCGA-B0-4846-01A
TCGA-B0-4846-11A
TCGA-B0-4847-01A
TCGA-B0-4847-11A
TCGA-B0-4848-01A
TCGA-B0-4848-11A
TCGA-B0-4849-01A
TCGA-B0-4849-11A
TCGA-B0-4852-01A
TCGA-B0-4852-11A
TCGA-B0-4945-01A
TCGA-B0-4945-11A
TCGA-B0-5080-01A
TCGA-B0-5080-11A
TCGA-B0-5083-01A
TCGA-B0-5083-11A
TCGA-B0-5092-01A
TCGA-B0-5092-11A
TCGA-B0-5094-01A
TCGA-B0-5094-11A
TCGA-B0-5095-01A
TCGA-B0-5095-11A
TCGA-B0-5096-01A
TCGA-B0-5096-11A
TCGA-B0-5097-01A
TCGA-B0-5097-11A
TCGA-B0-5098-01A
TCGA-B0-5098-11A
TCGA-B0-5099-01A
TCGA-B0-5099-11A
TCGA-B0-5100-01A
TCGA-B0-5100-11A
TCGA-B0-5102-01A
TCGA-B0-5102-11A
TCGA-B0-5104-01A
TCGA-B0-5104-11A
TCGA-B0-5106-01A
TCGA-B0-5106-11A
TCGA-B0-5107-01A
TCGA-B0-5107-11A
TCGA-B0-5108-01A
TCGA-B0-5108-11A
TCGA-B0-5109-01A
TCGA-B0-5109-11A
TCGA-B0-5110-01A
TCGA-B0-5110-11A
TCGA-B0-5113-01A
TCGA-B0-5113-11A
TCGA-B0-5115-01A
TCGA-B0-5115-11A
TCGA-B0-5116-01A
TCGA-B0-5116-11A
TCGA-B0-5117-01A
TCGA-B0-5117-11A
TCGA-B0-5119-01A
TCGA-B0-5119-11A
TCGA-B0-5120-01A
TCGA-B0-5120-11A
TCGA-B0-5121-01A
TCGA-B0-5121-11A
TCGA-B0-5399-01A
TCGA-B0-5400-01A
TCGA-B0-5400-11A
TCGA-B0-5402-01A
TCGA-B0-5402-11A
TCGA-B0-5690-01A
TCGA-B0-5691-01A
TCGA-B0-5692-01A
TCGA-B0-5693-01A
TCGA-B0-5694-01A
TCGA-B0-5695-01A
TCGA-B0-5696-01A
TCGA-B0-5697-01A
TCGA-B0-5698-01A
TCGA-B0-5699-01A
TCGA-B0-5700-01A
TCGA-B0-5701-01A
TCGA-B0-5702-01A
TCGA-B0-5703-01A
TCGA-B0-5705-01A
TCGA-B0-5706-01A
TCGA-B0-5707-01A
TCGA-B0-5709-01A
TCGA-B0-5710-01A
TCGA-B0-5710-11A
TCGA-B0-5711-01A
TCGA-B0-5711-11A
TCGA-B0-5712-01A
TCGA-B0-5712-11A
TCGA-B0-5713-01A
TCGA-B0-5713-11A
TCGA-B0-5812-01A
TCGA-B2-3924-01A
TCGA-B2-3924-01B
TCGA-B2-4101-01A
TCGA-B2-5633-01A
TCGA-B2-5633-01B
TCGA-B2-5635-01A
TCGA-B2-5635-01B
TCGA-B2-5636-01A
TCGA-B2-5639-01A
TCGA-B2-5641-01A
TCGA-B2-A4SR-01A
TCGA-B4-5377-01A
TCGA-B4-5378-01A
TCGA-B4-5832-01A
TCGA-B4-5834-01A
TCGA-B4-5835-01A
TCGA-B4-5836-01A
TCGA-B4-5838-01A
TCGA-B4-5843-01A
TCGA-B4-5844-01A
TCGA-B8-4146-01B
TCGA-B8-4153-01B
TCGA-B8-4621-01A
TCGA-B8-4622-01A
TCGA-B8-5158-01A
TCGA-B8-5159-01A
TCGA-B8-5162-01A
TCGA-B8-5163-01A
TCGA-B8-5164-01A
TCGA-B8-5165-01A
TCGA-B8-5545-01A
TCGA-B8-5546-01A
TCGA-B8-5549-01A
TCGA-B8-5550-01A
TCGA-B8-5551-01A
TCGA-B8-5552-01B
TCGA-B8-5553-01A
TCGA-B8-A54D-01A
TCGA-B8-A54E-01A
TCGA-B8-A54F-01A
TCGA-B8-A54G-01A
TCGA-B8-A54H-01A
TCGA-B8-A54I-01A
TCGA-B8-A54J-01A
TCGA-B8-A54K-01A
TCGA-B8-A7U6-01A
TCGA-B8-A8YJ-01A
TCGA-BP-4177-01A
TCGA-BP-4177-11A
TCGA-BP-4760-01A
TCGA-BP-4760-11A
TCGA-BP-4770-01A
TCGA-BP-4770-11A
TCGA-BP-4782-01A
TCGA-BP-4782-11A
TCGA-BP-4795-01A
TCGA-BP-4795-11A
TCGA-BP-4801-01A
TCGA-BP-4801-11A
TCGA-BP-4993-01A
TCGA-BP-4993-11A
TCGA-BP-5010-01A
TCGA-BP-5010-11A
TCGA-BP-5168-01A
TCGA-BP-5168-11A
TCGA-BP-5169-01A
TCGA-BP-5169-11A
TCGA-BP-5170-01A
TCGA-BP-5170-11A
TCGA-BP-5173-01A
TCGA-BP-5173-11A
TCGA-BP-5174-01A
TCGA-BP-5174-11A
TCGA-BP-5175-01A
TCGA-BP-5175-11A
TCGA-BP-5176-01A
TCGA-BP-5176-11A
TCGA-BP-5177-01A
TCGA-BP-5177-11A
TCGA-BP-5178-01A
TCGA-BP-5178-11A
TCGA-BP-5180-01A
TCGA-BP-5180-11A
TCGA-BP-5181-01A
TCGA-BP-5181-11A
TCGA-BP-5182-01A
TCGA-BP-5182-11A
TCGA-BP-5183-01A
TCGA-BP-5183-11A
TCGA-BP-5184-01A
TCGA-BP-5184-11A
TCGA-BP-5185-01A
TCGA-BP-5185-11A
TCGA-BP-5186-01A
TCGA-BP-5186-11A
TCGA-BP-5187-01A
TCGA-BP-5187-11A
TCGA-BP-5189-01A
TCGA-BP-5189-11A
TCGA-BP-5190-01A
TCGA-BP-5190-11A
TCGA-BP-5191-01A
TCGA-BP-5191-11A
TCGA-BP-5192-01A
TCGA-BP-5192-11A
TCGA-BP-5194-01A
TCGA-BP-5194-11A
TCGA-BP-5195-01A
TCGA-BP-5195-11A
TCGA-BP-5196-01A
TCGA-BP-5196-11A
TCGA-BP-5198-01A
TCGA-BP-5198-11A
TCGA-BP-5199-01A
TCGA-BP-5199-11A
TCGA-BP-5200-01A
TCGA-BP-5200-11A
TCGA-BP-5201-01A
TCGA-BP-5201-11A
TCGA-BP-5202-01A
TCGA-BP-5202-11A
TCGA-CJ-4869-01A
TCGA-CJ-4869-11A
TCGA-CJ-4882-01A
TCGA-CJ-4882-11A
TCGA-CJ-4897-01A
TCGA-CJ-4897-11A
TCGA-CJ-4901-01A
TCGA-CJ-4901-11A
TCGA-CJ-4902-01A
TCGA-CJ-4902-11A
TCGA-CJ-4903-01A
TCGA-CJ-4903-11A
TCGA-CJ-4904-01A
TCGA-CJ-4904-11A
TCGA-CJ-4905-01A
TCGA-CJ-4905-11A
TCGA-CJ-4907-01A
TCGA-CJ-4907-11A
TCGA-CJ-4908-01A
TCGA-CJ-4908-11A
TCGA-CJ-4912-01A
TCGA-CJ-4912-11A
TCGA-CJ-4913-01A
TCGA-CJ-4913-11A
TCGA-CJ-4916-01A
TCGA-CJ-4916-11A
TCGA-CJ-4918-01A
TCGA-CJ-4918-11A
TCGA-CJ-4920-01A
TCGA-CJ-4920-11A
TCGA-CJ-4923-01A
TCGA-CJ-4923-11A
TCGA-CJ-5671-01A
TCGA-CJ-5672-01A
TCGA-CJ-5675-01A
TCGA-CJ-5676-01A
TCGA-CJ-5677-01A
TCGA-CJ-5678-01A
TCGA-CJ-5679-01A
TCGA-CJ-5680-01A
TCGA-CJ-5681-01A
TCGA-CJ-5682-01A
TCGA-CJ-5683-01A
TCGA-CJ-5684-01A
TCGA-CJ-5686-01A
TCGA-CJ-5689-01A
TCGA-CJ-6027-01A
TCGA-CJ-6028-01A
TCGA-CJ-6030-01A
TCGA-CJ-6031-01A
TCGA-CJ-6032-01A
TCGA-CJ-6033-01A
TCGA-CW-5580-01A
TCGA-CW-5581-01A
TCGA-CW-5583-01A
TCGA-CW-5584-01A
TCGA-CW-5585-01A
TCGA-CW-5587-01A
TCGA-CW-5588-01A
TCGA-CW-5589-01A
TCGA-CW-5590-01A
TCGA-CW-5591-01A
TCGA-CW-6087-01A
TCGA-CW-6088-01A
TCGA-CW-6090-01A
TCGA-CW-6093-01A
TCGA-CW-6097-01A
TCGA-CZ-4853-01A
TCGA-CZ-4853-11A
TCGA-CZ-4856-01A
TCGA-CZ-4856-11A
TCGA-CZ-4859-01A
TCGA-CZ-4859-11A
TCGA-CZ-4863-01A
TCGA-CZ-4863-11A
TCGA-CZ-4864-01A
TCGA-CZ-4864-11A
TCGA-CZ-4865-01A
TCGA-CZ-4865-11A
TCGA-CZ-4866-01A
TCGA-CZ-4866-11A
TCGA-CZ-5451-01A
TCGA-CZ-5451-11A
TCGA-CZ-5452-01A
TCGA-CZ-5452-11A
TCGA-CZ-5453-01A
TCGA-CZ-5453-11A
TCGA-CZ-5454-01A
TCGA-CZ-5454-11A
TCGA-CZ-5455-01A
TCGA-CZ-5455-11A
TCGA-CZ-5456-01A
TCGA-CZ-5456-11A
TCGA-CZ-5457-01A
TCGA-CZ-5457-11A
TCGA-CZ-5458-01A
TCGA-CZ-5458-11A
TCGA-CZ-5459-01A
TCGA-CZ-5459-11A
TCGA-CZ-5460-01A
TCGA-CZ-5460-11A
TCGA-CZ-5461-01A
TCGA-CZ-5461-11A
TCGA-CZ-5462-01A
TCGA-CZ-5462-11A
TCGA-CZ-5463-01A
TCGA-CZ-5463-11A
TCGA-CZ-5464-01A
TCGA-CZ-5464-11A
TCGA-CZ-5465-01A
TCGA-CZ-5465-11A
TCGA-CZ-5466-01A
TCGA-CZ-5466-11A
TCGA-CZ-5467-01A
TCGA-CZ-5467-11A
TCGA-CZ-5468-01A
TCGA-CZ-5468-11A
TCGA-CZ-5469-01A
TCGA-CZ-5469-11A
TCGA-CZ-5470-01A
TCGA-CZ-5470-11A
TCGA-CZ-5982-01A
TCGA-CZ-5984-01A
TCGA-CZ-5985-01A
TCGA-CZ-5986-01A
TCGA-CZ-5987-01A
TCGA-CZ-5988-01A
TCGA-CZ-5989-01A
TCGA-DV-5565-01A
TCGA-DV-5566-01A
TCGA-DV-5567-01A
TCGA-DV-5568-01A
TCGA-DV-5569-01A
TCGA-DV-5573-01A
TCGA-DV-5574-01A
TCGA-DV-5575-01A
TCGA-DV-5576-01A
TCGA-DV-A4VX-01A
TCGA-DV-A4VZ-01A
TCGA-DV-A4W0-01A
TCGA-DV-A4W0-05A
TCGA-EU-5904-01A
TCGA-EU-5905-01A
TCGA-EU-5906-01A
TCGA-EU-5907-01A
TCGA-G6-A5PC-01A
TCGA-G6-A8L6-01A
TCGA-G6-A8L7-01A
TCGA-G6-A8L8-01A
TCGA-GK-A6C7-01A
TCGA-MM-A563-01A
TCGA-MM-A564-01A
TCGA-MM-A84U-01A
TCGA-MW-A4EC-01A
TCGA-T7-A92I-01A"""
